# 01 — Item Vector (TF-IDF with Weighted Soup)

Notebook 2/4 of the Content-Based pipeline.

**Strategy:**
- Build weighted soup: genres x3, author x2, title x1, description x1, shelves x1
- TF-IDF fit on random sample (300K texts) to limit RAM
- Transform full corpus in chunks, vstack sparse matrices

**Output:** `cb_tfidf_item_vector.npz`, `cb_tfidf_vectorizer.pkl`, `cb_soup_texts.parquet`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pyarrow as pa
import pyarrow.parquet as pq
import pickle, gc, os, re, psutil
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIG
# ============================================================
BASE_DIR   = '/content/drive/My Drive/Thesis/book_recsys'
DATA_DIR   = os.path.join(BASE_DIR, 'data/processed')

BOOKS_PQ   = os.path.join(DATA_DIR, 'cb_books.parquet')
SOUP_OUT   = os.path.join(DATA_DIR, 'cb_soup_texts.parquet')
TFIDF_VEC  = os.path.join(DATA_DIR, 'cb_tfidf_vectorizer.pkl')
TFIDF_MAT  = os.path.join(DATA_DIR, 'cb_tfidf_item_vector.npz')

SOUP_CHUNK   = 50_000   # rows per chunk when building soup
TFIDF_CHUNK  = 50_000   # rows per chunk when transforming
SAMPLE_SIZE  = 300_000  # sample size for TF-IDF fit
MAX_FEATURES = 20_000   # TF-IDF vocabulary size

print('Config done.')
print_ram()

Config done.
  [RAM] 0.23 GB


## 1. Build Weighted Soup

For each book, build a text string (soup) by repeating fields according to weight:
- `genres` x3 (highest signal)
- `author` x2
- `title` x1
- `description` x1
- `shelves` x1 (noise removed)

Text normalization: lowercase, remove special chars, hyphens to spaces, remove stop words.

In [ ]:
NOISE_SHELVES = {
    'to-read', 'currently-reading', 'owned', 'books-i-own',
    'favourites', 'favorite', 'default', 'to-buy', 'maybe',
    'wish-list', 'library', 'ebook', 'kindle'
}

def clean_text(text):
    """Normalize text: lowercase, remove special chars, hyphens to spaces."""
    if pd.isna(text) or str(text).strip() in ('', 'nan'):
        return ''
    t = str(text).lower()
    t = t.replace('-', ' ')
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def clean_shelves(shelves_str):
    """Clean shelves string, remove noise shelves."""
    if pd.isna(shelves_str) or str(shelves_str).strip() in ('', 'nan'):
        return ''
    parts = [clean_text(s.strip()) for s in str(shelves_str).split(',')]
    parts = [p for p in parts if p and p not in NOISE_SHELVES]
    return ' '.join(parts)

def make_soup(row):
    """Build weighted soup: genres x3, author x2, title x1, desc x1, shelves x1."""
    title   = clean_text(row.get('title', ''))
    desc    = clean_text(row.get('description', ''))
    genres  = clean_text(row.get('genres', ''))
    author  = clean_text(row.get('author', ''))
    shelves = clean_shelves(row.get('shelves', ''))

    parts = []
    if title:   parts.append(title)             # x1
    if desc:    parts.append(desc)              # x1
    for _ in range(3):                          # x3
        if genres: parts.append(genres)
    for _ in range(2):                          # x2
        if author: parts.append(author)
    if shelves: parts.append(shelves)           # x1
    return ' '.join(parts)

print('Soup functions ready.')

Soup functions ready.


In [ ]:
if os.path.exists(SOUP_OUT):
    print(f'SKIP: {os.path.basename(SOUP_OUT)} already exists.')
    n_books = pq.read_metadata(SOUP_OUT).num_rows
    print(f'  Total books: {n_books:,}')
else:
    print(f'Building soup, {SOUP_CHUNK:,} books per chunk...')
    pf = pq.ParquetFile(BOOKS_PQ)
    writer = None
    total = 0

    for batch in tqdm(pf.iter_batches(batch_size=SOUP_CHUNK)):
        chunk = batch.to_pandas()
        soup_df = pd.DataFrame({
            'book_id': chunk['book_id'],
            'soup': chunk.apply(make_soup, axis=1),
        })
        tbl = pa.Table.from_pandas(soup_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(SOUP_OUT, tbl.schema)
        writer.write_table(tbl)
        total += len(chunk)
        del chunk, soup_df, tbl
        gc.collect()

    if writer:
        writer.close()

    print(f'Soup created for {total:,} books.')
    print_ram('after soup')

Building soup, 50,000 books per chunk...


48it [07:44,  9.68s/it]

Soup created for 2,360,648 books.
  [RAM [after soup]] 0.74 GB


## 2. TF-IDF Fit (Sample-Based)

Random sample ~300K texts for fitting vocabulary. This avoids loading all soup texts into memory at once.

In [ ]:
if os.path.exists(TFIDF_VEC):
    print(f'SKIP: {os.path.basename(TFIDF_VEC)} already exists. Loading...')
    with open(TFIDF_VEC, 'rb') as f:
        tfidf = pickle.load(f)
    print(f'  Vocabulary size: {len(tfidf.vocabulary_):,}')
else:
    # Read all soup texts to sample from
    print(f'Sampling {SAMPLE_SIZE:,} texts for TF-IDF fit...')
    n_total = pq.read_metadata(SOUP_OUT).num_rows

    # Generate random indices
    rng = np.random.default_rng(42)
    sample_idx = set(rng.choice(n_total, size=min(SAMPLE_SIZE, n_total), replace=False))

    sample_texts = []
    offset = 0
    pf = pq.ParquetFile(SOUP_OUT)
    for batch in tqdm(pf.iter_batches(batch_size=SOUP_CHUNK, columns=['soup'])):
        chunk = batch.to_pandas()
        for i, text in enumerate(chunk['soup']):
            if (offset + i) in sample_idx:
                sample_texts.append(text if pd.notna(text) else '')
        offset += len(chunk)
        del chunk
        gc.collect()

    print(f'  Sampled {len(sample_texts):,} texts.')
    print_ram('after sampling')

    # Fit TF-IDF
    print('Fitting TF-IDF vectorizer...')
    tfidf = TfidfVectorizer(
        stop_words='english',
        max_features=MAX_FEATURES,
        ngram_range=(1, 2),
        dtype=np.float32,
        sublinear_tf=True,
        min_df=2,
        max_df=0.95,
    )
    tfidf.fit(sample_texts)
    print(f'  Vocabulary size: {len(tfidf.vocabulary_):,}')

    del sample_texts
    gc.collect()

    # Save vectorizer
    with open(TFIDF_VEC, 'wb') as f:
        pickle.dump(tfidf, f, protocol=4)
    print(f'  Saved: {os.path.basename(TFIDF_VEC)}')
    print_ram('after fit')

Sampling 300,000 texts for TF-IDF fit...


48it [00:28,  1.68it/s]


  Sampled 300,000 texts.
  [RAM [after sampling]] 1.05 GB
Fitting TF-IDF vectorizer...
  Vocabulary size: 20,000
  Saved: cb_tfidf_vectorizer.pkl
  [RAM [after fit]] 1.64 GB


## 3. TF-IDF Transform (Chunked)

Transform all soup texts in chunks. Each chunk produces a sparse sub-matrix. Final result: `scipy.sparse.vstack()` of all chunks → one CSR matrix.

In [ ]:
if os.path.exists(TFIDF_MAT):
    print(f'SKIP: {os.path.basename(TFIDF_MAT)} already exists.')
    tfidf_matrix = sp.load_npz(TFIDF_MAT)
    print(f'  Shape: {tfidf_matrix.shape}, nnz: {tfidf_matrix.nnz:,}')
else:
    print(f'Transforming in chunks of {TFIDF_CHUNK:,}...')
    sparse_chunks = []
    n_rows = 0

    pf = pq.ParquetFile(SOUP_OUT)
    for batch in tqdm(pf.iter_batches(batch_size=TFIDF_CHUNK, columns=['soup'])):
        chunk = batch.to_pandas()
        texts = chunk['soup'].fillna('').tolist()
        sub = tfidf.transform(texts)
        sparse_chunks.append(sub)
        n_rows += sub.shape[0]
        del chunk, texts
        gc.collect()

    print(f'  Vstacking {len(sparse_chunks)} chunks ({n_rows:,} rows)...')
    tfidf_matrix = sp.vstack(sparse_chunks, format='csr')
    del sparse_chunks
    gc.collect()

    # Save
    sp.save_npz(TFIDF_MAT, tfidf_matrix)
    nnz_mb = (tfidf_matrix.nnz * 4) / 1024**2  # float32 = 4 bytes
    print(f'  TF-IDF matrix: {tfidf_matrix.shape[0]:,} x {tfidf_matrix.shape[1]:,}')
    print(f'  nnz: {tfidf_matrix.nnz:,} (~{nnz_mb:.1f} MB)')
    print(f'  Saved: {os.path.basename(TFIDF_MAT)}')
    print_ram('after transform')

Transforming in chunks of 50,000...


48it [11:03, 13.83s/it]


  Vstacking 48 chunks (2,360,648 rows)...
  TF-IDF matrix: 2,360,648 x 20,000
  nnz: 169,379,014 (~646.1 MB)
  Saved: cb_tfidf_item_vector.npz
  [RAM [after transform]] 3.73 GB


## 4. Summary

In [ ]:
print('=' * 55)
print('  ITEM VECTOR SUMMARY')
print('=' * 55)
print(f'  Books in soup:     {pq.read_metadata(SOUP_OUT).num_rows:,}')
print(f'  TF-IDF shape:      {tfidf_matrix.shape}')
print(f'  TF-IDF nnz:        {tfidf_matrix.nnz:,}')
print(f'  Vocabulary size:   {len(tfidf.vocabulary_):,}')
print(f'  Max features:      {MAX_FEATURES:,}')
print('=' * 55)
print_ram('final')

  ITEM VECTOR SUMMARY
  Books in soup:     2,360,648
  TF-IDF shape:      (2360648, 20000)
  TF-IDF nnz:        169,379,014
  Vocabulary size:   20,000
  Max features:      20,000
  [RAM [final]] 3.73 GB
